# ML-04 — Verified FlyRank starter data contract

[Open in Colab](https://colab.research.google.com/github/Catacombs666/ML_int-/blob/main/work/notebooks/w03_data_contract.ipynb)

Built on the FlyRank ML Internship dataset — https://flyrank.ai. This notebook publishes aggregate or derived diagnostics only.

In [1]:
import os, sys, subprocess, importlib.util
from pathlib import Path
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/ML_int-')]
ROOT=next((p for p in candidates if (p/'data/raw/content_refresh_anonymized.csv').is_file()),None)
if ROOT is None:
    destination=Path('/content/ML_int-') if Path('/content').exists() else Path.cwd()/'ML_int-'
    subprocess.run(['git','clone','https://github.com/Catacombs666/ML_int-.git',str(destination)],check=True)
    ROOT=destination
if os.environ.get('CAPSTONE_OFFLINE') != '1':
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'work/requirements.txt')],check=True)
sys.path.insert(0,str(ROOT/'work/scripts'))
import json, pandas as pd, numpy as np
import ctr_capstone as study
print('Setup ready: public starter data; no token required; numerical dependencies loaded.')


Setup ready: public starter data; no token required; numerical dependencies loaded.


## Unit and window

One row = pseudonymized content item over the supplied trailing 90-day snapshot. Absolute endpoints are not recorded.

In [2]:
d,audit=study.read_data()
print(json.dumps(audit,indent=2))

{
  "raw_rows": 30000,
  "raw_clients": 32,
  "raw_columns": 44,
  "duplicate_content_rows": 0,
  "excluded_below_500": 13274,
  "excluded_position_after_volume": 0,
  "eligible_rows": 16726,
  "eligible_clients": 28,
  "position_zero_raw": 1205,
  "ctr_rounding_max_pp": 0.005000000000000782,
  "sha256": "c43bdac4eccfa17fcd8a33974fe36f2c998c03a3ae3af8d80cf712abab5d6396",
  "source_git_blob": "2f2f41fdb07868dc5599449ce9d817f58f3b2ae9",
  "source_commit": "3b657dda40c1b668072a53371883fd4413d60f08",
  "date_window": "Trailing 90-day snapshot; absolute export/end dates not present in the starter CSV. Warehouse dates are not assigned to this slice.",
  "missing_pct_eligible": {
    "word_count": 28.393,
    "search_volume": 1.602,
    "cpc": 1.602,
    "competition": 1.602,
    "main_intent": 1.262
  }
}


## Roles and exclusions

IDs are context; clicks and impressions define the label and measurement weights. CTR, clicks, trend and engagement outcomes never enter the feature vector.

In [3]:
print('Features:',study.FEATURES)
print('Excluded label/ID fields:',sorted(study.BANNED))
assert not set(study.FEATURES)&study.BANNED
print('Role audit passed.')

Features: ['avg_position', 'log_word_count', 'content_age_days', 'days_since_last_update', 'log_search_volume', 'competition', 'log_cpc', 'content_type', 'main_intent']
Excluded label/ID fields: ['ai_traffic_pct', 'clicks_90d', 'clicks_last_30d', 'clicks_prev_30d', 'client_id', 'content_id', 'ctr', 'engagement_rate', 'scroll_rate', 'sessions_90d', 'trend_direction', 'trend_pct']
Role audit passed.


## Grain and coverage

Apply the 500-impression floor and positive measured-position guard. Missing metadata is handled inside fitted pipelines.

In [4]:
idx,counts=study.split_clients(d)
print(json.dumps(counts,indent=2))
assert sum(x['rows'] for x in counts.values())==len(d)
print('Eligible cohort and grouped split reconcile.')

{
  "train": {
    "rows": 8483,
    "clients": 17
  },
  "validation": {
    "rows": 4330,
    "clients": 5
  },
  "test": {
    "rows": 3913,
    "clients": 6
  }
}
Eligible cohort and grouped split reconcile.


## Data limits

The warehouse and its calendar endpoints are not used. Query text, SERP features, and experiment outcomes are unavailable.

In [5]:
print(audit['date_window'])
print('No row dump is published; table summaries only.')

Trailing 90-day snapshot; absolute export/end dates not present in the starter CSV. Warehouse dates are not assigned to this slice.
No row dump is published; table summaries only.
